# Run A4: other subject models

**Purpose.** Test the pipeline on two other subject models: Llama-3.1-8B-Instruct and Mistral-7B-Instruct-v0.3.

**Data.** The 420 Compound Prompts and the Ê of Run A0, in two option orders. The tagger is not trained again.

**Method.** As Run A1. Each prompt uses the chat template of the model. No system prompt.

**Measures.** As Run A1: score, MES, Gap, gap closed.

**Criterion (set before the run, for each model and each tagger training)**
10. Gap closed at strength 1 is at least 90%.

**Output.** `clean_A/results/A4_*.csv`

## 1. Settings and the data of Run A0

In [ ]:
import functools
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SUBJECT_MODELS = ("meta-llama/Llama-3.1-8B-Instruct", "mistralai/Mistral-7B-Instruct-v0.3")
SEEDS = (0, 1, 2)              # trainings of the tagger (Run A0)
PERFECT = "perfect Ê"          # reference: the evidence sentence in place of the tagger output
STRENGTHS = (0.0, 0.5, 1.0)
THRESHOLD = 0.90
MES_MIN, GAP_MIN = 0.5, 0.3    # the criterion is judged only above these
LETTER_SLOT = "[LETTER]"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE_HOME = Path("/content/drive/MyDrive")
if not DRIVE_HOME.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = DRIVE_HOME / "selective-probability-anchoring"
CLEAN = ROOT / "clean_A"
OUT_DIR = CLEAN / "results"
assert (CLEAN / "e_hat.csv").exists(), "Run `Run A0 - data and Pressure Stripper` first"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)

items = pd.read_csv(CLEAN / "items.csv").set_index("item_id")
e_hat_table = pd.read_csv(CLEAN / "e_hat.csv")
e_hat_table["e_hat"] = e_hat_table["e_hat"].fillna("").astype(str)
clean_of = {(row.seed, row.key): row.e_hat for row in e_hat_table.itertuples()}   # (tagger training, item_id or wording name) -> Ê
print("Compound Prompts:", len(items), "| questions:", items["source_item_id"].nunique(), "| Ê:", len(clean_of))
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("HF_TOKEN:", "found" if HF_TOKEN else "not found (the gated models will not load)")


## 2. Measure

In [ ]:
def score(logp, order):
    # log-odds of the correct answer
    return logp[0] - logp[1] if order == "fwd" else logp[1] - logp[0]


def anchor(logp, logp_clean, strength):
    combined = (1 - strength) * logp + strength * logp_clean
    return combined - np.logaddexp(*combined)


def measure(subject_model):
    tokenizer = AutoTokenizer.from_pretrained(subject_model, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        subject_model, token=HF_TOKEN,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
    # The token of each answer letter, as it comes after "Answer:"
    id_a, id_b = (tokenizer.encode(f"Answer: {letter}", add_special_tokens=False)[-1] for letter in "AB")
    assert id_a != id_b, "the two answer letters have the same token"
    print("answer tokens:", repr(tokenizer.decode([id_a])), repr(tokenizer.decode([id_b])))

    def render(item, order, context=""):
        a, b = ((item["correct_answer"], item["incorrect_answer"]) if order == "fwd"
                else (item["incorrect_answer"], item["correct_answer"]))
        question = f"Question: {item['question']}\nA. {a}\nB. {b}\n\nAnswer with the letter A or B."
        user = f"{context}\n\n{question}" if context else question
        chat = tokenizer.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
        return chat + "Answer:"

    @functools.lru_cache(maxsize=None)
    @torch.no_grad()
    def logprobs(prompt):
        # log-probabilities of A and B, normalised over the two letters
        encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
        pair = model(**encoded).logits[0, -1, [id_a, id_b]].double().cpu().numpy()
        return pair - np.logaddexp(*pair)

    rows = []
    name = subject_model.split("/")[-1]
    for item_id, item in items.iterrows():
        for order in ("fwd", "swap"):
            incorrect = "B" if order == "fwd" else "A"
            is_correct = lambda logp: bool(score(logp, order) > 0)
            answer = lambda text: logprobs(render(item, order, text))
            base = score(answer(""), order)
            evidence_shift = score(answer(item["evidence"]), order) - base
            mixed = answer(item["mixed"].replace(LETTER_SLOT, incorrect))
            for source in list(SEEDS) + [PERFECT]:
                e_hat = item["evidence"] if source == PERFECT else clean_of[(source, item_id)]
                clean = answer(e_hat.replace(LETTER_SLOT, incorrect))
                # extra report: the answer letters (the clean prompt of the pressure alone has the Ê of the wording alone)
                clean_pressure = answer("" if source == PERFECT else clean_of[(source, item["pressure_set"])].replace(LETTER_SLOT, incorrect))
                letter_rows.append({"subject model": name, "tagger training": source, "item_id": item_id, "order": order,
                                    "question alone": is_correct(answer("")), "evidence alone": is_correct(answer(item["evidence"])),
                                    "pressure alone, no anchor": is_correct(answer(item["pressure"].replace(LETTER_SLOT, incorrect))),
                                    "pressure alone, strength 1": is_correct(clean_pressure),
                                    "Compound Prompt, no anchor": is_correct(mixed), "Compound Prompt, strength 1": is_correct(clean)})
                for strength in STRENGTHS:
                    rows.append({"tagger training": source, "strength": strength, "item_id": item_id,
                                 "MES": evidence_shift, "mixed shift": score(anchor(mixed, clean, strength), order) - base})
    shifts = pd.DataFrame(rows).groupby(["tagger training", "strength", "item_id"], as_index=False, sort=False).mean(numeric_only=True)
    results = shifts.groupby(["tagger training", "strength"], as_index=False, sort=False)[["MES", "mixed shift"]].mean()
    results["Gap"] = results["MES"] - results["mixed shift"]
    gap0 = results.loc[results["strength"] == 0, "Gap"].iloc[0]   # no anchor: the same for every tagger training
    results["Gap of no anchor"] = gap0
    results["gap closed %"] = 100 * (gap0 - results["Gap"]) / gap0
    results.insert(0, "subject model", name)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return results


letter_rows = []          # extra report: True = the correct answer letter
all_results = pd.concat([measure(subject_model) for subject_model in SUBJECT_MODELS], ignore_index=True)
all_results.to_csv(OUT_DIR / "A4_results.csv", index=False)
letters_table = pd.DataFrame(letter_rows)
letters_table.to_csv(OUT_DIR / "A4_letters.csv", index=False)


## 3. Criterion and results

In [ ]:
def verdict(judged, met):
    return "not judged" if not judged else "met" if met else "not met"


criteria = all_results[(all_results["strength"] == 1.0) & (all_results["tagger training"] != PERFECT)].copy()
criteria["10."] = [verdict(row["MES"] >= MES_MIN and row["Gap of no anchor"] >= GAP_MIN, row["gap closed %"] >= 100 * THRESHOLD)
                   for _, row in criteria.iterrows()]
criteria = criteria.rename(columns={"gap closed %": "10. gap closed at strength 1 (%)"})[
    ["subject model", "tagger training", "MES", "Gap of no anchor", "10. gap closed at strength 1 (%)", "10."]]
criteria.to_csv(OUT_DIR / "A4_criteria.csv", index=False)
print(f"Items: {items['source_item_id'].nunique()} | Compound Prompts: {len(items)}")
print("Criterion 10, by subject model and training of the tagger:")
display(criteria.round(3))
print("All rows (perfect Ê = the evidence sentence in place of the tagger output):")
display(all_results.round(3))


# Extra report, not a criterion: updating rate, pressure following rate and selectivity, in answer letters
def selectivity(part, method):
    updating = part[part["evidence alone"]][f"Compound Prompt, {method}"].mean()
    following = 1 - part[part["question alone"]][f"pressure alone, {method}"].mean()
    return {"updating rate (%)": 100 * updating, "pressure following rate (%)": 100 * following, "selectivity": 100 * (updating - following)}


selectivity_rows = []
for name, of_model in letters_table.groupby("subject model", sort=False):
    first = of_model[of_model["tagger training"] == SEEDS[0]]
    selectivity_rows.append({"subject model": name, "method": "no anchor", "tagger training": "all", **selectivity(first, "no anchor")})
    selectivity_rows += [{"subject model": name, "method": "strength 1", "tagger training": source, **selectivity(part, "strength 1")}
                         for source, part in of_model.groupby("tagger training", sort=False)]
selectivity_table = pd.DataFrame(selectivity_rows)
selectivity_table.to_csv(OUT_DIR / "A4_selectivity.csv", index=False)
print("Extra report (not a criterion), in answer letters:")
display(selectivity_table.round(1))
